# Probabilistic imputation: are virtual cells calibrated under cohort shift? (`future_plan.md` §18.6)

**Question.** The cell models so far return one value per cell. Given a predictive *distribution* per cell, is it calibrated within a cohort, and does that hold across cohorts?

**Grid** (as `train-test-grid`, replicate-0 split and seeds): train on {Jackson, Danenberg, Both} × test on {held-out Jackson patients, the 130 frozen Danenberg test patients}; 11 shared markers, LightGBM only.

**Predictive distributions per cell:**
- **Point** LightGBM (L2), as before. CRPS of a point = absolute error.
- **Quantile** LightGBM: 9 levels (0.025, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.975); sorted per cell (no crossing); 50 draws by inverse-CDF interpolation.
- **Residual sampling:** point prediction + residuals of *validation* patients of the training cohort, drawn within the cell's prediction decile; 50 draws.
- **Climatology:** 50 draws from the training cohort's marginal of the target marker (the distributional constant). Plus the point constant (training mean).

**Spaces (no leakage of the target marker in transfer):**
- Scoring is on arcsinh(x/5) of the test cohort's measured value (Danenberg: per-cell image means; Jackson: published table).
- Own and Both: inputs harmonised with the test cohort's statistics, outputs mapped back with them (the marker was measured in that cohort's training data).
- Transfer: test inputs scaled with the *training* cohort's statistics, outputs mapped back with the training cohort's statistics of the target marker (as the main figure's `[X-scaled inputs]` versions).

**Metrics** (10 markers, ERα reported but left out of means; 1000× patient bootstrap): CRPS; coverage of central 50/80/95% intervals; mean 80% width; PIT KS distance and 10-bin histogram; exact W2 between one draw per cell and the truth; SD ratio; R4 of the point / median prediction.

**Also:** per-marker histograms of the training targets and 1–3 component Gaussian mixture BIC, to decide the §18.6.1 mixture likelihood.

**Decision (§0.3 item 2):** if quantile LightGBM is calibrated under transfer (|cov80 − 0.80| < 0.03 for ≥ 8/10 markers in both directions), the Bayesian arm's H1 has little room and it runs for few-shot (S3) only.

In [ ]:
import os, glob, json, time, sys, subprocess, warnings, traceback
import numpy as np, pandas as pd
import pyarrow.parquet as pq
from joblib import Parallel, delayed
try:
    import lightgbm as lgb
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lightgbm'], check=True)
    import lightgbm as lgb
from sklearn.mixture import GaussianMixture
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

IN = os.environ.get("IN_ROOT", "/kaggle/input")
OUT = os.environ.get("OUT_DIR", "/kaggle/working")
SMOKE = os.environ.get("SMOKE") == "1"
SEED = 20261013
REP = 0
N_BOOT = 100 if SMOKE else 1000
MIN_CELLS = 200
COF = 5.0
N_FIT_FINAL = 30_000 if SMOKE else 300_000    # cells per cohort for the point model (as train-test-grid)
N_FIT_Q = 20_000 if SMOKE else 100_000        # cells per cohort for each quantile model
N_DRAW = 50
QL = np.array([0.025, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.975])
LGB = dict(n_estimators=40 if SMOKE else 250, learning_rate=0.06, num_leaves=31, min_child_samples=50,
           subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, n_jobs=os.cpu_count(), verbose=-1)
LGBQ = dict(LGB, n_estimators=40 if SMOKE else 150)
NUC = ["histone_h3", "dna1", "dna2"]
D_CHAN = ['histone_h3', 'sma', 'cytokeratin_5', 'cd38', 'hla_dr', 'cytokeratin_8_18', 'cd15', 'fsp1', 'cd163',
          'cd278_icos', 'cd134', 'cd68', 'c_erb_b_2_her2', 'cd3', 'podoplanin', 'cd11c', 'cd279_pd_1', 'gitr_tnfrsf18',
          'cd16', 'c_erb_b_2_her2_d8f12', 'cd45ra', 'beta_2_microglobulin', 'cd45', 'foxp3', 'cd20',
          'estrogen_receptor_alpha', 'cd8a', 'cd57', 'ki_67', 'cd140b_pdgf_receptor_beta', 'caveolin_1', 'cd4',
          'cd31_v_wf', 'cxcl12_sdf_1', 'hla_abc', 'pan_cytokeratin', 'cleaved_caspase3', 'dna1', 'dna2']
J_CHAN = ['histone_h3', 'histone_h3_trimethylate', 'cytokeratin_5', 'fibronectin', 'cytokeratin_19', 'cytokeratin_8_18',
          'twist', 'cd68', 'keratin_14_krt14', 'sma', 'vimentin', 'c_myc', 'c_erb_b_2_her2', 'cd3', 'histone_h3_phospho',
          'slug', 'estrogen_receptor_alpha', 'progesterone_receptor_a_b', 'p53', 'cd44', 'cd45', 'gata3', 'cd20',
          'carbonic_anhydrase_ix', 'e_cadherin_p_cadherin', 'ki_67', 'egfr', 's6', 'v_wf', 'm_tor', 'cytokeratin_7',
          'pan_cytokeratin', 'cleaved_parp', 'dna1', 'dna2']
MARKERS = sorted(m for m in set(D_CHAN) & set(J_CHAN) if m not in NUC)
assert len(MARKERS) == 11, MARKERS
EXCL = "estrogen_receptor_alpha"
SOURCES = ["Jackson", "Danenberg", "Both"]
COHORTS = ["Jackson", "Danenberg"]
OTHER = {"Jackson": "Danenberg", "Danenberg": "Jackson"}
T0 = time.time()

def tick(msg):
    print(f"[{(time.time() - T0) / 60:6.1f} min] {msg}", flush=True)

def g(pattern):
    return sorted(glob.glob(f"{IN}/**/{pattern}", recursive=True))

def near(name, anchor):
    hits = g(name)
    assert hits, f"missing input {name}"
    return max(hits, key=lambda c: len(os.path.commonpath([c, anchor])))

FAILED = []
def guard(name):
    def deco(fn):
        try:
            return fn()
        except Exception:
            print(f"!!! block {name} failed:\n{traceback.format_exc()}", flush=True)
            FAILED.append(name)
            return None
    return deco

## Cell tables and Danenberg per-cell image means (as `train-test-grid`)

In [ ]:
P = {}
for p in g("cells.parquet"):
    key = "danenberg" if "metabric_id" in pq.read_schema(p).names else "jackson"
    P[key] = dict(cells=p)
P["danenberg"]["split"] = near("danenberg_patient_split.csv", P["danenberg"]["cells"])

def load_table(key):
    cells = pd.read_parquet(P[key]["cells"])
    if key == "danenberg":
        cells = cells[(cells.is_tumour == 1) & (cells.is_hot_aggregate == 0)]
        pid = "metabric_id"
    else:
        cells = cells[cells.is_tumour == 1]
        pid = "patient_id"
    n_per = cells.groupby("image_name").size()
    cells = cells[cells.image_name.isin(n_per.index[n_per >= MIN_CELLS])]
    cells = cells.sort_values(["image_name", "object_number"]).reset_index(drop=True)
    img_code, img_names = pd.factorize(cells.image_name, sort=True)
    starts = np.r_[0, np.flatnonzero(np.diff(img_code)) + 1, len(cells)]
    return dict(cells=cells, img=img_code, img_names=np.asarray(img_names), starts=starts,
                patient=cells[pid].astype(str).values,
                log_area=np.log(np.maximum(cells["area"].values, 1)).astype(np.float32))

DD, DJ = load_table("danenberg"), load_table("jackson")
split = pd.read_csv(P["danenberg"]["split"]).set_index("metabric_id").split
DD["split"] = pd.Series(DD["patient"]).map(split).fillna("none").values
tick(f"tables: Danenberg {len(DD['cells']):,} cells, Jackson {len(DJ['cells']):,} cells")

if subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--only-binary=:all:", "imagecodecs==2024.9.22"]).returncode:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--only-binary=:all:", "imagecodecs"], check=True)
import tifffile
tifs = g("MB_*.tiff")
D_IMG = {os.path.splitext(os.path.basename(p))[0]: p for p in tifs if "/masks/" not in p.replace("\\", "/")}
D_MASK = {os.path.splitext(os.path.basename(p))[0]: p for p in tifs if "/masks/" in p.replace("\\", "/")}
OBJ = DD["cells"].object_number.values.astype(np.int64)
ST = DD["starts"]

def raw_means(img_path, mask_path, obj):
    m = tifffile.imread(mask_path).astype(np.int64)
    lut = np.full(max(int(m.max()), int(obj.max())) + 1, -1, np.int64)
    lut[obj] = np.arange(len(obj))
    loc = lut[m.ravel()]
    v = np.flatnonzero(loc >= 0)
    lv = loc[v]
    cnt = np.bincount(lv, minlength=len(obj))
    im = tifffile.imread(img_path)
    if im.ndim == 3 and im.shape[0] == len(D_CHAN) and im.shape[-1] != len(D_CHAN):
        im = np.moveaxis(im, 0, -1)
    im = im.reshape(-1, im.shape[-1]).astype(np.float32)
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.stack([np.where(cnt > 0, np.bincount(lv, im[v, c], len(obj)) / np.maximum(cnt, 1), np.nan)
                         for c in range(im.shape[1])], 1).astype(np.float32)

RAW = np.vstack(Parallel(n_jobs=os.cpu_count())(delayed(raw_means)(D_IMG[s], D_MASK[s], OBJ[a:b])
                                                for s, a, b in zip(DD["img_names"], ST[:-1], ST[1:])))
assert RAW.shape == (len(DD["cells"]), len(D_CHAN))
dix = {m: i for i, m in enumerate(D_CHAN)}
IN_SET = NUC + MARKERS
X = {"Danenberg": np.c_[np.arcsinh(RAW[:, [dix[c] for c in IN_SET]] / COF), DD["log_area"]].astype(np.float64),
     "Jackson": np.c_[np.arcsinh(DJ["cells"][IN_SET].values.astype(np.float64) / COF), DJ["log_area"]].astype(np.float64)}
del RAW
tick("per-cell image means")

## Splits (replicate 0 of `train-test-grid`) and per-cohort harmonisation statistics (as in `train-test-grid`: all cells of each cohort)

In [ ]:
jp, jcode = np.unique(DJ["patient"], return_inverse=True)
u = np.random.default_rng(SEED + 11 + REP).random(len(jp))
J_SPLIT = np.where(u < 0.7, "train", np.where(u < 0.8, "val", "test"))[jcode]
D_TRVA_PAT = np.unique(DD["patient"][DD["split"] != "test"])
uv = np.random.default_rng(SEED + 21 + REP).random(len(D_TRVA_PAT))
dval = set(D_TRVA_PAT[uv < 0.125])
D_SPLIT = np.where(DD["split"] == "test", "test", np.where(pd.Series(DD["patient"]).isin(dval).values, "val", "train"))
SPLIT = {"Jackson": J_SPLIT, "Danenberg": D_SPLIT}
IDX = {c: {s: np.flatnonzero(SPLIT[c] == s) for s in ("train", "val", "test")} for c in COHORTS}
PAT = {"Jackson": DJ["patient"], "Danenberg": DD["patient"]}
IMG = {"Jackson": DJ["img"], "Danenberg": DD["img"]}
for c in COHORTS:
    print(c, {s: (len(v), len(set(PAT[c][v]))) for s, v in IDX[c].items()}, "(cells, patients)")

# harmonisation statistics, exactly as harm() in train-test-grid (all cells of the cohort)
STAT = {}
for c in COHORTS:
    med = np.nanmedian(X[c], 0)
    q05, q95 = np.nanpercentile(X[c], [5, 95], 0)
    STAT[c] = (med, np.where(q95 - q05 > 1e-3, q95 - q05, np.nanstd(X[c], 0) + 1e-6))

def H(c_data, c_stat):
    med, sc = STAT[c_stat]
    return ((X[c_data] - med) / sc).astype(np.float32)

HH = {(cd, cs): H(cd, cs) for cd in COHORTS for cs in COHORTS}
tick("splits and harmonisation")

## Training-target histograms and mixture BIC (decides the §18.6.1 likelihood)

In [ ]:
@guard("histograms")
def _():
    rows, fig = [], plt.figure(figsize=(16, 9))
    for j, m in enumerate(MARKERS):
        k = IN_SET.index(m)
        ax = fig.add_subplot(3, 4, j + 1)
        for c, col in (("Jackson", "#2a78d6"), ("Danenberg", "#008300")):
            y = X[c][IDX[c]["train"], k]
            y = y[np.isfinite(y)]
            ax.hist(y, bins=120, density=True, histtype="step", color=col, lw=1.2, label=c)
            s = np.random.default_rng(SEED).choice(y, min(50_000, len(y)), replace=False)[:, None]
            bics = [GaussianMixture(nc, random_state=SEED, n_init=2).fit(s).bic(s) for nc in (1, 2, 3)]
            g2 = GaussianMixture(2, random_state=SEED, n_init=2).fit(s)
            lo = int(np.argmin(g2.means_.ravel()))
            rows.append(dict(cohort=c, marker=m, n=len(y), frac_at_min=float(np.mean(y <= y.min() + 1e-6)),
                             frac_below_0p1=float(np.mean(y < 0.1)), bic1=bics[0], bic2=bics[1], bic3=bics[2],
                             best_k=int(np.argmin(bics)) + 1, low_comp_weight=float(g2.weights_[lo]),
                             low_comp_mean=float(g2.means_.ravel()[lo]), high_comp_mean=float(g2.means_.ravel()[1 - lo]),
                             q01=float(np.quantile(y, 0.01)), q50=float(np.median(y)), q99=float(np.quantile(y, 0.99))))
        ax.set_xlabel(f"{m}, arcsinh(x/5)")
        ax.set_ylabel("density")
        ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(f"{OUT}/target_histograms.png", dpi=150)
    R = pd.DataFrame(rows)
    R.to_csv(f"{OUT}/target_mixture_bic.csv", index=False)
    print(R.round(3).to_string(index=False))
tick("histograms")

## Scoring helpers (CRPS from draws, coverage, PIT, W2, R4; patient bootstrap)

In [ ]:
def crps_draws(S, y):
    # S: (n, k) draws, y: (n,). CRPS = E|X - y| - 0.5 E|X - X'|, the unbiased ("fair") estimator over draws
    k = S.shape[1]
    Ss = np.sort(S, 1)
    t1 = np.abs(Ss - y[:, None]).mean(1)
    w = (2 * np.arange(1, k + 1) - k - 1) / (k * (k - 1))
    t2 = (Ss * w).sum(1)          # = 0.5 E|X - X'|
    return t1 - t2

def boot_mean(v, pc, n_pat, W):
    s, c = np.bincount(pc, v, n_pat), np.bincount(pc, minlength=n_pat).astype(np.float64)
    return float(s.sum() / c.sum()), (W @ s) / (W @ c)

def r4(a, b, img):
    ok = np.isfinite(a) & np.isfinite(b)
    a, b, im = a[ok], b[ok], img[ok]
    k = int(im.max()) + 1
    cnt = np.maximum(np.bincount(im, minlength=k), 1)
    a = a - (np.bincount(im, a, k) / cnt)[im]
    b = b - (np.bincount(im, b, k) / cnt)[im]
    return float((a * b).sum() / np.sqrt((a * a).sum() * (b * b).sum()))

def interp_draws(Q, rng, n):
    # inverse-CDF draws from per-cell quantiles Q (cells × levels), linear between levels; tails held flat
    # at the outer levels beyond 0.025 / 0.975 (i.e. the outer 5% is clamped)
    uq = rng.random((Q.shape[0], n))
    out = np.empty_like(uq)
    for i in range(n):
        u_ = np.clip(uq[:, i], QL[0], QL[-1])
        j = np.clip(np.searchsorted(QL, u_) - 1, 0, len(QL) - 2)
        t = (u_ - QL[j]) / (QL[j + 1] - QL[j])
        r = np.arange(Q.shape[0])
        out[:, i] = Q[r, j] * (1 - t) + Q[r, j + 1] * t
    return out

TEST = {c: IDX[c]["test"] for c in COHORTS}
BOOTW = {}
for c in COHORTS:
    tp, tcode = np.unique(PAT[c][TEST[c]], return_inverse=True)
    W = np.random.default_rng(SEED + (12 if c == "Jackson" else 13)).multinomial(
        len(tp), np.full(len(tp), 1 / len(tp)), size=N_BOOT).astype(np.float64)
    BOOTW[c] = (tcode, len(tp), W)

## Fit and score: 3 training sets × 11 markers; each scored on both test cohorts

In [ ]:
ROWS, PER_MARKER_PIT, LOG = [], [], []
BOOTS = {}

def mapping(src, test):
    # (input harmonisation cohort, output back-mapping cohort) for a training set and test cohort
    if src == "Both" or src == test:
        return test, test
    return src, src            # transfer: everything in the training cohort's scale (no target-marker leakage)

def score(src, test, m, name, draws=None, point=None):
    k = IN_SET.index(m)
    y = X[test][TEST[test], k]
    ok = np.isfinite(y)
    tcode, n_pat, W = BOOTW[test]
    pc = tcode[ok]
    y = y[ok]
    rec = dict(train=src, test=test, marker=m, model=name, n_cells=int(ok.sum()))
    if draws is not None:
        S = draws[ok]
        crps = crps_draws(S, y)
        Ss = np.sort(S, 1)
        for lev in (0.5, 0.8, 0.95):
            lo_i, hi_i = (1 - lev) / 2, 1 - (1 - lev) / 2
            lo, hi = np.quantile(S, lo_i, 1), np.quantile(S, hi_i, 1)
            cov = ((y >= lo) & (y <= hi)).astype(np.float64)
            pt, bv = boot_mean(cov, pc, n_pat, W)
            rec[f"cov{int(lev * 100)}"], rec[f"cov{int(lev * 100)}_lo"], rec[f"cov{int(lev * 100)}_hi"] = pt, *np.percentile(bv, [2.5, 97.5])
            BOOTS[(src, test, m, name, f"cov{int(lev * 100)}")] = (pt, bv)
            if lev == 0.8:
                rec["width80"] = float(np.mean(hi - lo))
        rng = np.random.default_rng(SEED + 7)
        below = (Ss < y[:, None]).sum(1)
        ties = (Ss == y[:, None]).sum(1)
        pit = (below + rng.random(len(y)) * (ties + 1)) / (S.shape[1] + 1)
        hist = np.histogram(pit, bins=10, range=(0, 1))[0] / len(pit)
        PER_MARKER_PIT.append(dict(train=src, test=test, marker=m, model=name, **{f"b{i}": h for i, h in enumerate(hist)}))
        ps = np.sort(pit)
        rec["pit_ks"] = float(np.max(np.abs(ps - (np.arange(1, len(ps) + 1) - 0.5) / len(ps))))
        one = S[np.arange(len(y)), np.random.default_rng(SEED + 9).integers(0, S.shape[1], len(y))]
        rec["W2_draw"] = float(np.sqrt(np.mean((np.sort(one) - np.sort(y)) ** 2)))
        rec["sd_ratio_draw"] = float(one.std() / y.std())
        med = np.median(S, 1)
        rec["R4_median"] = r4(y, med, IMG[test][TEST[test]][ok])
    else:
        crps = np.abs(point[ok] - y)
        rec["W2_draw"] = float(np.sqrt(np.mean((np.sort(point[ok]) - np.sort(y)) ** 2)))
        rec["sd_ratio_draw"] = float(point[ok].std() / y.std())
        if np.std(point[ok]) > 0:
            rec["R4_median"] = r4(y, point[ok], IMG[test][TEST[test]][ok])
    pt, bv = boot_mean(crps, pc, n_pat, W)
    rec["crps"], (rec["crps_lo"], rec["crps_hi"]) = pt, np.percentile(bv, [2.5, 97.5])
    BOOTS[(src, test, m, name, "crps")] = (pt, bv)
    ROWS.append(rec)

for src in SOURCES:
    tick(f"train on {src}")
    for m in MARKERS:
        k = IN_SET.index(m)
        cols = [i for i in range(len(IN_SET) + 1) if i != k]
        # training rows: harmonised with their own cohort's statistics (as train-test-grid)
        parts = COHORTS if src == "Both" else [src]
        rs = np.random.default_rng(SEED + REP)
        Xtr, ytr, Xtq, ytq = [], [], [], []
        for c in parts:
            Hc = HH[(c, c)]
            tr = IDX[c]["train"][np.isfinite(Hc[IDX[c]["train"], k])]
            f1 = np.sort(rs.choice(tr, min(N_FIT_FINAL, len(tr)), replace=False))
            fq = np.sort(rs.choice(tr, min(N_FIT_Q, len(tr)), replace=False))
            Xtr.append(Hc[f1][:, cols]); ytr.append(Hc[f1, k])
            Xtq.append(Hc[fq][:, cols]); ytq.append(Hc[fq, k])
        Xtr, ytr, Xtq, ytq = np.vstack(Xtr), np.concatenate(ytr), np.vstack(Xtq), np.concatenate(ytq)
        t0 = time.time()
        point = lgb.LGBMRegressor(random_state=SEED + REP, **LGB).fit(Xtr, ytr)
        qms = [lgb.LGBMRegressor(objective="quantile", alpha=float(a), random_state=SEED + REP, **LGBQ).fit(Xtq, ytq)
               for a in QL]
        LOG.append(dict(train=src, marker=m, seconds=time.time() - t0, n_point=len(ytr), n_quantile=len(ytq)))
        # residual pools from validation patients of the training cohort(s), in harmonised units, by prediction decile
        rp, rr = [], []
        for c in parts:
            Hc = HH[(c, c)]
            va = IDX[c]["val"][np.isfinite(Hc[IDX[c]["val"], k])]
            rp.append(point.predict(Hc[va][:, cols])); rr.append(Hc[va, k] - rp[-1])
        rp, rr = np.concatenate(rp), np.concatenate(rr)
        edges = np.quantile(rp, np.linspace(0, 1, 11)[1:-1])
        dec = np.searchsorted(edges, rp)
        pools = [rr[dec == d] for d in range(10)]
        # climatology pool: the training cohort's own training values of the marker (harmonised units)
        clim = ytr
        for test in COHORTS:
            cin, cout = mapping(src, test)
            Hin = HH[(test, cin)][TEST[test]][:, cols]
            med_o, sc_o = STAT[cout][0][k], STAT[cout][1][k]
            back = lambda v: v * sc_o + med_o
            pp = point.predict(Hin)
            rng = np.random.default_rng(SEED + 31)
            # point model and point constant
            score(src, test, m, "Point LightGBM", point=back(pp))
            score(src, test, m, "Constant (training mean)", point=np.full(len(pp), back(clim.mean())))
            # quantile LightGBM
            Q = np.sort(np.stack([q.predict(Hin) for q in qms], 1), 1)
            score(src, test, m, "Quantile LightGBM", draws=back(interp_draws(Q, rng, N_DRAW)))
            # residual sampling
            d = np.searchsorted(edges, pp)
            Sres = np.empty((len(pp), N_DRAW))
            for dd in range(10):
                ii = np.flatnonzero(d == dd)
                if len(ii):
                    Sres[ii] = pp[ii, None] + rng.choice(pools[dd], (len(ii), N_DRAW))
            score(src, test, m, "Residual sampling", draws=back(Sres))
            # climatology (distributional constant)
            score(src, test, m, "Climatology", draws=back(rng.choice(clim, (len(pp), N_DRAW))))
        tick(f"  {src}: {m}")
    pd.DataFrame(ROWS).to_csv(f"{OUT}/scores_per_marker.csv", index=False)
pd.DataFrame(LOG).to_csv(f"{OUT}/training_log.csv", index=False)
pd.DataFrame(PER_MARKER_PIT).to_csv(f"{OUT}/pit_histograms.csv", index=False)

## Summaries: means over 10 markers (ERα excluded), paired CRPS differences, the §0.3 decision

In [ ]:
R = pd.DataFrame(ROWS)
R10 = R[R.marker != EXCL]
def setting(r):
    return "own" if r.train == r.test else ("both" if r.train == "Both" else "transfer")
R10 = R10.assign(setting=R10.apply(setting, axis=1))
cols = ["crps", "cov50", "cov80", "cov95", "width80", "pit_ks", "W2_draw", "sd_ratio_draw", "R4_median"]
SUM = R10.groupby(["test", "setting", "train", "model"])[cols].mean().reset_index()
SUM.to_csv(f"{OUT}/summary_mean10.csv", index=False)
with pd.option_context("display.width", 250, "display.max_columns", 20):
    print(SUM.round(3).to_string(index=False))

# mean-over-markers bootstrap CIs for CRPS and coverage, and paired CRPS differences vs Point and Climatology
ms10 = [m for m in MARKERS if m != EXCL]
AGG, DIFF = [], []
for (src, test) in [(s, t) for s in SOURCES for t in COHORTS]:
    for name in ["Point LightGBM", "Quantile LightGBM", "Residual sampling", "Climatology", "Constant (training mean)"]:
        for met in ["crps", "cov80", "cov95"]:
            keys = [(src, test, m, name, met) for m in ms10]
            if not all(k_ in BOOTS for k_ in keys):
                continue
            bv = np.mean([BOOTS[k_][1] for k_ in keys], 0)
            AGG.append(dict(train=src, test=test, model=name, metric=met, value=np.mean([BOOTS[k_][0] for k_ in keys]),
                            lo=np.percentile(bv, 2.5), hi=np.percentile(bv, 97.5)))
        for ref in ["Point LightGBM", "Climatology"]:
            if ref == name:
                continue
            for m in ms10:
                a, b = BOOTS[(src, test, m, name, "crps")], BOOTS[(src, test, m, ref, "crps")]
                lo, hi = np.percentile(a[1] - b[1], [2.5, 97.5])
                DIFF.append(dict(train=src, test=test, model=name, vs=ref, marker=m, diff=a[0] - b[0], lo=lo, hi=hi,
                                 call="better" if hi < 0 else "worse" if lo > 0 else "n.s."))
AGG, DIFF = pd.DataFrame(AGG), pd.DataFrame(DIFF)
AGG.to_csv(f"{OUT}/summary_mean10_ci.csv", index=False)
DIFF.to_csv(f"{OUT}/crps_differences.csv", index=False)
print(AGG.pivot_table(index=["test", "train", "model"], columns="metric", values="value").round(3).to_string())
print(DIFF.groupby(["test", "train", "model", "vs"]).call.value_counts().unstack(fill_value=0).to_string())

# §0.3 item 2 decision: quantile LightGBM calibrated under transfer?
tr = R10[(R10.model == "Quantile LightGBM") & (R10.train != "Both") & (R10.train != R10.test)]
calib = tr.assign(ok=(tr.cov80 - 0.80).abs() < 0.03).groupby("test").ok.sum()
decision = dict(n_markers_calibrated_80=calib.to_dict(),
                quantile_calibrated_under_transfer=bool((calib >= 8).all()),
                rule="|cov80 - 0.80| < 0.03 for >= 8/10 markers in both directions")
print(decision)
json.dump(decision, open(f"{OUT}/decision.json", "w"), indent=1)

## Figure: CRPS and 80% coverage by setting

In [ ]:
@guard("figure")
def _():
    names = ["Point LightGBM", "Quantile LightGBM", "Residual sampling", "Climatology"]
    colr = {"Point LightGBM": "#52514e", "Quantile LightGBM": "#2a78d6", "Residual sampling": "#008300", "Climatology": "#b0aea7"}
    fig, axes = plt.subplots(2, 2, figsize=(12, 7.5))
    for i, test in enumerate(COHORTS):
        order = [test, "Both", OTHER[test]]
        labs = [f"own ({test})", "both", f"transfer ({OTHER[test]}-trained)"]
        for j, met in enumerate(["crps", "cov80"]):
            ax = axes[j, i]
            for q, nm in enumerate(names):
                xs, vs, los, his = [], [], [], []
                for p_, src in enumerate(order):
                    r = AGG[(AGG.train == src) & (AGG.test == test) & (AGG.model == nm) & (AGG.metric == met)]
                    if len(r):
                        xs.append(p_ + (q - 1.5) * 0.18); vs.append(r.value.iloc[0]); los.append(r.lo.iloc[0]); his.append(r.hi.iloc[0])
                if xs:
                    vs, los, his = map(np.array, (vs, los, his))
                    ax.bar(xs, vs, width=0.16, color=colr[nm], label=nm)
                    ax.errorbar(xs, vs, yerr=[vs - los, his - vs], fmt="none", ecolor="#0b0b0b", elinewidth=0.8, capsize=2)
            if met == "cov80":
                ax.axhline(0.8, color="#0b0b0b", lw=1, ls=(0, (4, 3)))
                ax.set_ylim(0, 1)
            ax.set_xticks(range(3)); ax.set_xticklabels(labs)
            ax.set_ylabel(("CRPS, arcsinh units (lower is better)" if met == "crps" else "80% interval coverage") + f"\ntest: {test}")
            ax.spines[["top", "right"]].set_visible(False)
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=4, frameon=False)
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    fig.savefig(f"{OUT}/fig_prob_impute.png", dpi=200)

json.dump({"failed_blocks": FAILED, "minutes": (time.time() - T0) / 60}, open(f"{OUT}/run_info.json", "w"))
print("failed blocks:", FAILED if FAILED else "none")